# Chapter 5: Using AI with Manim

Manim has a large API, and remembering every class and method is hard. AI assistants can do the heavy lifting: **generate** code from a description, and **debug** it when something breaks.

We'll use two tools, each with two sections (4 in total):
1. **5.1 Generating code with Gemini**
2. **5.2 Debugging with Gemini**
3. **5.3 Generating code with opencode**
4. **5.4 Debugging with opencode**

Both tools follow the same pattern — only the location differs: Gemini runs in your browser, opencode runs in your terminal.

In [ ]:
from manim import *
config.media_width = "75%"
config.verbosity = "WARNING"

## 5.1 Generating code with Gemini

[Gemini](https://google.gemini.com) runs in your browser. A good prompt says **"Manim Community Edition"** (to avoid the old API), describes the scene, and asks for a `Scene` class.

> Using **Manim Community Edition**, write a `Scene` called `PlotParabola` that plots $f(x)=x^2$ on axes from -5 to 5, draws the curve in blue, and animates a yellow dot traveling along it from $x=-3$ to $x=3$.

In [ ]:
%%manim -qm PlotParabola

class PlotParabola(Scene):
    def construct(self):
        axes = Axes(
            x_range=[-5, 5, 1],
            y_range=[0, 10, 2],
            x_length=8,
            y_length=5,
            axis_config={"include_numbers": True},
        ).to_edge(DOWN)

        curve = axes.plot(lambda x: x**2, x_range=[-3, 3], color=BLUE)
        label = MathTex("f(x) = x^2", color=YELLOW).next_to(curve, UP)
        dot = Dot(color=YELLOW).move_to(axes.c2p(-3, (-3) ** 2))

        self.play(Create(axes), run_time=1.5)
        self.play(Create(curve), Write(label), run_time=2)
        self.play(MoveAlongPath(dot, curve, run_time=4, rate_func=linear))
        self.wait(1)
        self.play(FadeOut(axes, curve, label, dot))
        self.wait()

### Exercise 5.1

Open [gemini.com](https://google.gemini.com) and ask it to generate a Manim CE scene called `Shapes` that places a **circle**, a **square**, and a **triangle** side by side, each in a different color, and fades them in one at a time. Paste the result into the cell below and run it.

In [ ]:
%%manim -qm Shapes

class Shapes(Scene):
    def construct(self):
        # TODO: paste the scene generated by Gemini here
        pass

## 5.2 Debugging with Gemini

When a scene errors, paste the code **and the full traceback** into Gemini and ask it to fix it. The scene below is **intentionally bugged** — run it to see the error.

In [ ]:
%%manim -qm BuggedScene

class BuggedScene(Scene):
    def construct(self):
        circle = Circle(color=BLUE)
        square = Square(color=RED)
        self.play(Create(circle))
        self.wait()
        # Oops! There's a typo below. Can you spot it?
        self.play(Transfrom(circle, square))
        self.wait()

You'll get `NameError: name 'Transfrom' is not defined`. Tell Gemini *"This Manim CE scene raises this error, fix it"* — it spots that `Transfrom` is a misspelling of **`Transform`**:

In [ ]:
%%manim -qm FixedScene

class FixedScene(Scene):
    def construct(self):
        circle = Circle(color=BLUE)
        square = Square(color=RED)
        self.play(Create(circle))
        self.wait()
        # Fixed: 'Transform' was misspelled as 'Transfrom'
        self.play(Transform(circle, square))
        self.wait()

### Exercise 5.2

The scene below fails with `NameError: name 'play' is not defined`. Paste the code and the error into Gemini, apply its fix, and make the scene run.

In [ ]:
%%manim -qm DebugMe

class DebugMe(Scene):
    def construct(self):
        square = Square(color=GREEN)
        play(Create(square))  # Bug: something is missing here
        self.wait()

## 5.3 Generating code with opencode

[opencode](https://opencode.ai) works **just like Gemini**, but it runs in your terminal and can read and edit your files directly. Open a terminal in your project, run `opencode`, and prompt it the same way. It can even render the scene for you with `manim -qm file.py ClassName`.

> Using **Manim Community Edition**, write a `Scene` called `CountingScene` that shows the numbers 1 to 5 one after another, each in a different color, scaling up briefly. Then render it.

In [ ]:
%%manim -qm CountingScene

class CountingScene(Scene):
    def construct(self):
        colors = [RED, ORANGE, YELLOW, GREEN, BLUE]
        for i in range(1, 6):
            number = Tex(str(i), font_size=96, color=colors[i - 1])
            self.play(Write(number), run_time=0.6)
            self.play(number.animate.scale(1.3), run_time=0.3)
            self.play(number.animate.scale(1 / 1.3), run_time=0.3)
            self.play(FadeOut(number, shift=UP * 0.5), run_time=0.3)
        self.wait()

### Exercise 5.3

In a terminal, run `opencode` and ask it to generate a Manim CE scene called `SineWave` that plots $y = \sin(x)$ on axes from $-2\pi$ to $2\pi$ and animates the curve being drawn. Have opencode render it, then paste the scene below.

In [ ]:
%%manim -qm SineWave

class SineWave(Scene):
    def construct(self):
        # TODO: paste the scene generated by opencode here
        pass

## 5.4 Debugging with opencode

Debugging works the same as with Gemini — but **better**: opencode can read your scene file and the error logs directly, so you don't have to copy-paste everything. Just tell it *"my render failed, fix it"*.

The scene below uses 3D camera calls but inherits from the wrong class — run it to see the error.

In [ ]:
%%manim -qm Bugged3D

class Bugged3D(Scene):  # Bug: should be ThreeDScene
    def construct(self):
        cube = Cube(side_length=2, fill_opacity=0.8, color=BLUE)
        self.set_camera_orientation(phi=75 * DEGREES, theta=45 * DEGREES)
        self.play(Create(cube))
        self.wait()

You'll get `AttributeError: 'Scene' object has no attribute 'set_camera_orientation'`. opencode reads the traceback, sees the camera calls, and fixes it by inheriting from **`ThreeDScene`** instead of `Scene`:

In [ ]:
%%manim -qm Fixed3D

class Fixed3D(ThreeDScene):  # Fixed: inherit ThreeDScene for 3D content
    def construct(self):
        cube = Cube(side_length=2, fill_opacity=0.8, color=BLUE)
        self.set_camera_orientation(phi=75 * DEGREES, theta=45 * DEGREES)
        self.play(Create(cube))
        self.move_camera(theta=135 * DEGREES, run_time=3)
        self.wait()

### Exercise 5.4

The scene below fails with `NameError: name 'Fadein' is not defined`. Tell opencode your render failed and let it fix the file, then run the corrected scene.

In [ ]:
%%manim -qm DebugMe2

class DebugMe2(Scene):
    def construct(self):
        title = Text("Fixed with opencode", font_size=64)
        self.play(Fadein(title))  # Bug: wrong capitalization
        self.wait()

## 5.5 Using opencode with this Jupyter notebook

So far we've treated opencode like a code generator. But because this whole tutorial lives in a **Jupyter notebook** (`.ipynb`), opencode can do something Gemini cannot: **edit the notebook file directly**.

A `.ipynb` file is just a list of cells stored as JSON. opencode understands that structure, so it can **add new cells**, **fix the code inside a cell**, or **run a cell** for you — with no copy-pasting on your part.

### Workflow

1. Open the notebook in Jupyter in your browser.
2. In a **second terminal** in the same folder, run `opencode`.
3. Ask opencode to do something to the notebook, for example:
   - *"Add a new code cell with a Manim scene that animates a rotating square."*
   - *"Fix the bug in the `DebugMe2` cell."*
   - *"Add a `%%manim -qm` cell that plots $y=\cos(x)$."*
4. After opencode saves its changes, **reload the notebook in Jupyter** (*File → Revert Notebook to Saved*, or close and reopen) so the new cells appear.

> Tip: Jupyter keeps its own copy in memory. **Save in Jupyter before** asking opencode to edit, and **reload after** — otherwise one may overwrite the other.

### Exercise 5.5

With this notebook open in Jupyter, open a second terminal next to it and run `opencode`. Ask it: *"Add a new code cell to this notebook containing a Manim CE scene called `BouncingBall` that moves a `Dot` up and down."* Then revert/reload the notebook in Jupyter to see the new cell, and run it.


## Summary

Both **Gemini** and **opencode** are Manim pair-programmers: a **clear prompt** generates a scene, and the **traceback** fixes it. Use Gemini for quick web brainstorming, and opencode to generate, edit, and render directly inside your project.